Notebook 1: Configuração Inicial e Imports

In [1]:
import re
from pathlib import Path
import pandas as pd

caminho_projeto = Path.cwd()
if caminho_projeto.name == "Notebook_limpeza_dados":
    caminho_projeto = caminho_projeto.parent

caminho_entrada = (
    caminho_projeto / "Datasets Items de casa" / "Aquecedores-de-agua-e-gas.csv"
)
caminho_saida = (
    caminho_projeto
    / "Datasets Items de casa"
    / "aquecedores-de-agua-e-gas_limpo.csv"
)

Notebook 2: Carga e Higienização Inicial

In [2]:
df = pd.read_csv(caminho_entrada, encoding="utf-8")
df.columns = df.columns.str.replace("ï»¿", "").str.strip()
df = df.rename(columns={"Classificação  PBE": "Classificação PBE"})
df = df.drop_duplicates()

Notebook 3: Recuperação de Dados via Regex

In [3]:
def extrair_vazao(modelo):
    match = re.search(r"(\d{2})\s*L", str(modelo), re.IGNORECASE)
    return float(match.group(1)) if match else None

def extrair_gas(modelo):
    mod = str(modelo).upper()
    if "GLP" in mod:
        return "GLP"
    if " GN" in mod or "GN " in mod or mod.endswith("GN"):
        return "GN"
    return None

df["Capacidade de Vazão (litros/min)"] = df[
    "Capacidade de Vazão (litros/min)"
].fillna(df["Modelo"].apply(extrair_vazao))

df["Tipo de Gás"] = df["Tipo de Gás"].fillna(df["Modelo"].apply(extrair_gas))

Notebook 4: Imputação por Média de Dispositivos Semelhantes

In [4]:
cols_tecnicas = [
    "Potência (kW)",
    "Rendimento (%)",
    "Consumo Máximo de Gás GN (m3/h) GLP (kg/h)",
]

for col in cols_tecnicas:
    media_marca_vazao = df.groupby(["Marca", "Capacidade de Vazão (litros/min)"])[
        col
    ].transform("mean")
    df[col] = df[col].fillna(media_marca_vazao)

    media_vazao_geral = df.groupby("Capacidade de Vazão (litros/min)")[
        col
    ].transform("mean")
    df[col] = df[col].fillna(media_vazao_geral)

Notebook 5: Validação de Qualidade e Exportação 

In [5]:
cols_tecnicas = [
    "Potência (kW)",
    "Rendimento (%)",
    "Consumo Máximo de Gás GN (m3/h) GLP (kg/h)",
]

for col in cols_tecnicas:
    media_marca_vazao = df.groupby(["Marca", "Capacidade de Vazão (litros/min)"])[
        col
    ].transform("mean")
    df[col] = df[col].fillna(media_marca_vazao)

    media_vazao_geral = df.groupby("Capacidade de Vazão (litros/min)")[
        col
    ].transform("mean")
    df[col] = df[col].fillna(media_vazao_geral)

Notebook 6: Validação de Qualidade e Exportação

In [ ]:
colunas_calculo_obrigatorias = [
    "Capacidade de Vazão (litros/min)",
    "Tipo de Gás",
    "Consumo Máximo de Gás GN (m3/h) GLP (kg/h)",
    "Potência (kW)",
    "Rendimento (%)",
]

df_limpo = df.dropna(subset=colunas_calculo_obrigatorias).copy()

if "Código de Barras" in df_limpo.columns:
    df_limpo["Código de Barras"] = (
        df_limpo["Código de Barras"]
        .fillna("N/A")
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )

df_limpo.to_csv(caminho_saida, index=False, encoding="utf-8")
print(f"✓ Dataset gerado com sucesso em: {caminho_saida}")


KeyboardInterrupt: 